# ПР-6. Классификация русскоязычных текстов
**Проект:** CryptoMind AI | **Выполнил:** <Баргыбаев Нуртилек, cs32 группа>

Корпус: 600 новостей Lenta.ru (стриминг с HuggingFace, см. `src/fetch_corpus.py`),
4 темы по 150 документов: Экономика, Спорт, Наука и техника, Культура.
Разметка: тема, присвоенная редактором издания — считаем эталонной.
Задача: по тексту новости предсказать тему.

In [8]:
import pandas as pd

corpus = pd.read_csv("../data/processed/lenta_corpus.csv")
print(corpus["topic"].value_counts())
corpus.head(3)

topic
Наука и техника    150
Культура           150
Спорт              150
Экономика          150
Name: count, dtype: int64


,text,topic
0,Astrium займется ликвидацией французского химо...,Наука и техника
1,У симулятора Minecraft появился LEGO-конкурент...,Наука и техника
2,В США отложен запуск военного спутника из-за н...,Наука и техника


## 1. Предобработка русского текста
Нижний регистр → токенизация регулярным выражением → удаление стоп-слов
(nltk, русский список) → лемматизация (pymorphy3, словарная база —
«бежали» → «бежать»). Оставляем только кириллицу.

In [9]:
import re
import nltk
import pymorphy3

nltk.download("stopwords")
stop = set(nltk.corpus.stopwords.words("russian"))
morph = pymorphy3.MorphAnalyzer()

def preprocess(text):
    words = re.findall(r"[а-я]+", text.lower())
    return " ".join(morph.parse(w)[0].normal_form
                    for w in words if w not in stop and len(w) > 2)

corpus["text_clean"] = corpus["text"].apply(preprocess)   # 1-2 минуты, дождись!
corpus[["text", "text_clean"]].head(3)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,text_clean
0,Astrium займется ликвидацией французского химо...,заняться ликвидация французский химоружие мини...
1,У симулятора Minecraft появился LEGO-конкурент...,симулятор появиться конкурент компания анонсир...
2,В США отложен запуск военного спутника из-за н...,сша отложить запуск военный спутник неполадка ...


## 2. Два алгоритма на TF-IDF
LogReg (линейная, интерпретируема) vs MultinomialNB (классика для текстов).
Векторизатор: TF-IDF, униграммы + биграммы. Разбиение стратифицированное 80/20.
Метрика: macro F1.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score

Xtr, Xte, ytr, yte = train_test_split(
    corpus["text_clean"], corpus["topic"],
    test_size=0.2, random_state=42, stratify=corpus["topic"])

models = {
    "LogReg": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                        ("clf", LogisticRegression(max_iter=1000))]),
    "NaiveBayes": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                            ("clf", MultinomialNB())]),
}
for name, m in models.items():
    m.fit(Xtr, ytr)
    print(f"\n=== {name} | macro F1 = "
          f"{f1_score(yte, m.predict(Xte), average='macro'):.3f} ===")
    print(classification_report(yte, m.predict(Xte), zero_division=0))


=== LogReg | macro F1 = 0.941 ===
                 precision    recall  f1-score   support

       Культура       0.94      0.97      0.95        30
Наука и техника       0.96      0.83      0.89        30
          Спорт       1.00      1.00      1.00        30
      Экономика       0.88      0.97      0.92        30

       accuracy                           0.94       120
      macro avg       0.94      0.94      0.94       120
   weighted avg       0.94      0.94      0.94       120


=== NaiveBayes | macro F1 = 0.925 ===
                 precision    recall  f1-score   support

       Культура       0.93      0.90      0.92        30
Наука и техника       0.96      0.83      0.89        30
          Спорт       1.00      1.00      1.00        30
      Экономика       0.83      0.97      0.89        30

       accuracy                           0.93       120
      macro avg       0.93      0.93      0.93       120
   weighted avg       0.93      0.93      0.93       120



In [11]:
pred = models["LogReg"].predict(Xte)
mask = pred != yte
print(f"Ошибок: {mask.sum()} из {len(yte)}")
for truth, p, txt in list(zip(yte[mask], pred[mask], Xte[mask]))[:5]:
    print(f"правда: {truth} | предсказано: {p}\n{txt[:250]}\n---")

Ошибок: 7 из 120
правда: Культура | предсказано: Наука и техника
польша обнаружить фреска век фреска обнаружить польский православный церковь деревня посад рыботыцк расположить восток страна учёный датировать век сообщать такой заявление сделать вторник февраль время пресс конференция посвятить дать открытие истор
---
правда: Наука и техника | предсказано: Культура
китай заявить кража статуя будда мумия монах внутри статуя будда внутри который нидерландский учёный обнаружить мумия монах украсть китайский деревня янчунь год заявить житель населить пункт пекин намеренный добиваться возвращение артефакт сообщать с
---
правда: Наука и техника | предсказано: Экономика
уйти куратор железный разработка корпорация уйти боб мансфилд старший вица президент компания аппаратный проектирование сообщаться пресс релиз опубликовать сайт это произойти несколько месяц мансфилд возглавлять аппаратный разработка компьютер год см
---
правда: Наука и техника | предсказано: Экономика
заработать миллиард долл

## Выводы
- **Лучшая модель: LogReg, macro F1 = 0.941** против 0.925 у NaiveBayes.
  Разница невелика, но устойчива: линейная модель лучше разделяет пересекающуюся
  лексику, тогда как у Байеса независимость признаков (слова и биграммы
  коррелированы) работает хуже.
- **Самая чистая тема — «Спорт» (F1 = 1.00)**: наиболее изолированная лексика.
  Самые низкие F1 у «Наука и техника» (0.89 у обеих моделей) — она пересекается
  с двумя темами сразу.
- **Наиболее путаная пара: «Наука и техника» ↔ «Экономика».** Из 7 ошибок 4 —
  именно она: новости про корпорации (выручка Apple, уход вице-президента)
  набиты экономической лексикой — «миллиард», «выручка», «прибыль», «корпорация»
  — и уходят в «Экономику». Это не ошибка разметки, а объективный стык тем:
  IT-бизнес лексически ближе к экономике, чем к науке.
- **Вторая пара ошибок: «Культура» ↔ «Наука и техника»** — археология
  (фрески, мумия в статуе Будды) лежит на границе: «учёные», «датировка»
  тянут в науку, «храм», «артефакт» — в культуру.
- **О корпусе:** 600 документов (по 150 на тему) оказалось достаточно для
  устойчивых оценок: разброс F1 между темами объясняется их лексической
  близостью, а не нехваткой данных. Разметка редакторская — эталонная,
  ручная разметка не требовалась. Пайплайн (лемматизация + TF-IDF + биграммы)
  переносим на любую русскоязычную тематическую классификацию, в том числе
  на отбор рыночно-значимых новостей в рамках CryptoMind AI.